# 09D · Native squares, segmented cells and local RNA leakage

**Question.** In the original Ada-14R evaluation region, how sparse are the 2‑µm squares, how often does an 8‑µm block span cell boundaries, and how much lineage signal survives a leakage correction on the same segmented cells?

The default reads every Space Ranger in-tissue square in the region, including measured zeros. It retrieves the historical QC flag separately. Background evidence comes from square centers outside every original Proseg mask. An uncovered square may contain genuine cellular RNA, including RNA from a missed cell. The historical segmentation used QC-retained transcripts. The background table reports how much out-of-mask RNA comes from outside that QC subset.

**First run.** Extract the package beside the existing 09 notebooks. In a JupyterLab terminal in that folder run `bash setup_09d_sparkle.sh`, then select **CT26 SPARKLE 0.1.4 (CPU)** and choose **Run All**. The installer uses Python 3.11 or newer for this pinned environment. Set `CT26_PYTHON=/path/to/python3.11` before the command if your terminal uses an older Python. R 4.5.2 can stay in your usual kernel.

The original Step 00 store is `/stash/data/nonclin/TBIO-8110_VisiumHD-Adagrasib-mouseTumor/zarr_files/Ada-14R.zarr`, table `square_002um`. Step 00 exported its barcode alignment and QC flags to Parquet. This notebook combines those with Space Ranger raw counts and positions and can run when the Zarr itself is unmounted. Step 03 provides the square-center to Proseg-cell assignment. A missing or incompatible assignment cache triggers local reconstruction from the original Step 02 polygons.

In [ ]:
from pathlib import Path
from IPython.display import display, Image
from run09d import run_pilot
from review09 import show_review_zip

RESULT_ROOT = Path("09_pilot_results")
RUN_DIR = None  # By default use the completed Ada-14R 09B run in the current or recorded folder.
STUDY_ROOT = Path("/stash/data/nonclin/TBIO-8110_VisiumHD-Adagrasib-mouseTumor")
TEMP_PROJECT_ROOT = Path("/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8110/tmp")
PATH_OVERRIDES = {}  # For individually moved inputs, e.g. {"square_dir": Path("/mount/square_002um")}.
DOMAIN = "all_in_tissue"  # Alternative for sensitivity: "qc_retained_in_tissue".
RUN_SPARKLE = True       # False gives the full sparsity, boundary and observed-marker audit.
PADDING_UM = 400.
BACKGROUND_GENES = 128

## Read the 09B region and compare its measurement units

Counts use the original full feature axis for sparsity. An 8‑µm block must contain all 16 eligible 2‑µm positions in the core. The companion row of 2‑µm constituents covers precisely the same footprint. The cell rows report both hard square-center assignment and the observed prepared Proseg matrix. Cell membership is held fixed for paired correction comparisons. The region outside the original core supplies source cells and background observations for fitting.

The targeted fit adds the prespecified lineage markers to the 128 genes with the most out-of-mask counts. It estimates one decay scale on this buffered region and fits gene-specific leakage, subject to the package's fit criterion. Count estimates after correction may be fractional. The CPU workflow streams raw MEX if raw HDF5 is unavailable. On the full Ada-14R files the initial scan can take time. Each run writes a new folder.

In [ ]:
out, archive, status = run_pilot(result_root=RESULT_ROOT, run_dir=RUN_DIR,
                                 study_root=STUDY_ROOT, temp_project_root=TEMP_PROJECT_ROOT,
                                 overrides=PATH_OVERRIDES, domain=DOMAIN,
                                 run_sparkle=RUN_SPARKLE, padding_um=PADDING_UM,
                                 background_genes=BACKGROUND_GENES)
print("Run status:", status)
print("Local output:", out)
review_download = show_review_zip(archive)

## Native sparsity, mask coverage and marker retention

Read the unit counts, zero fractions and median UMIs together. A complete 8‑µm block can capture more RNA while crossing more than one biological cell. The two mixing fractions can overlap. Compare the all-in-tissue 2‑µm row to the historical QC subset to assess selection by the older mask. A substantial loss of core cells during bin-to-cell matching, a large cell-centroid offset, or broad gaps in the background map warrant inspection before interpreting a fit.

In [ ]:
import json
import pandas as pd

def show_json(filename):
    p = out / filename
    if p.is_file():
        print(json.dumps(json.loads(p.read_text()), indent=2))

if (out / "sparsity.csv").is_file():
    display(pd.read_csv(out / "sparsity.csv").round(3))
    show_json("coverage_summary.json")
if (out / "native_sparsity_and_background.png").is_file():
    display(Image(filename=str(out / "native_sparsity_and_background.png")))
if (out / "traceback.txt").is_file():
    print((out / "traceback.txt").read_text())

## Decide whether the correction earns a larger pilot

Compare observed and corrected counts from the same hard masks. The observed 09B matrix is a separate reference for the effect of changing the assignment rule. The `panT09C` group uses the original observed two-distinct-pan-T-gene criterion and includes cells with any prior annotation. Near (≤25 µm) and far (≥50 µm) refer to centroid distance from a previously labeled macrophage/monocyte in the 09B region. Use the non-macrophage and T-labeled subgroups when testing proximity, since a macrophage has zero distance to its own centroid. The earlier labels and these distances are proxies. Inspect the counts of available cells for every group.

A promising pattern would be lower myeloid burden in pan-T-supported cells near macrophages together with retention of TCR/CD3 in those cells and myeloid genes in source macrophages. Check depth, represented area and B-cell group sizes in `panel_comparisons.csv`. Loss of native lineage signal in small cells, few informative out-of-mask fits, or widespread subtraction across all groups would argue against extending this pilot. Similar overall UMAP shapes can occur under any of these outcomes.

In [ ]:
if (out / "panel_comparisons.csv").is_file():
    results = pd.read_csv(out / "panel_comparisons.csv")
    focus = results.group.isin(["panT09C", "panT09C_nonmac_near25", "panT09C_nonmac_far50", "T_panT09C_near25", "T_panT09C_far50", "T_label", "macrophage_label", "B_label"])
    selected = results.loc[focus & results.panel.isin(["panT", "myeloid", "B"])]
    display(selected.pivot(index=["group","n_cells","panel"], columns="method", values="mean_panel_counts").round(3))
if (out / "marker_retention.png").is_file():
    display(Image(filename=str(out / "marker_retention.png")))
show_json("sparkle_diagnostics.json")

The **return-to-chat ZIP** linked above contains the aggregate tables, figures, path and environment receipts, and any recorded error. The cell-level table and targeted count matrix stay in the local run folder. The archive also works after an audit-only run. Native counts and original segmentations are read-only.

[SPARKLE methods and input contract](https://github.com/WangShuai-3/SPARKLE) · [Python distribution 0.1.4](https://pypi.org/project/stambient/0.1.4/)